# LlamaIndex: базовый RAG на документах РобоТех

Пройдем путь от файла до ответа на тех же документах, что и в `rag_workshop.ipynb`.

Загрузим документы, разобьем их на nodes, создадим индекс и сравним поиск с фильтром и без него. Используем четыре HR-документа: два архивных за 2024 год и два за 2026 год.

## Подготовка

Из корня репозитория выполните `uv sync`. Откройте этот ноутбук в VS Code и выберите ядро Python из `.venv`. Запускайте ячейки сверху вниз.

Для генерации нужна запущенная Ollama и модель:

```bash
ollama pull hf.co/Qwen/Qwen3-4B-GGUF:Q4_K_M
```

При первом запуске `BAAI/bge-m3` скачается из Hugging Face; индексация может занять несколько минут. Для раздела с Qdrant дополнительно нужен Docker.

In [1]:
import os
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown

from llama_index.core import (
    SimpleDirectoryReader,  # Читает файлы и создает документы
    Settings,  # Общие настройки моделей LlamaIndex
    VectorStoreIndex,  # Индекс для поиска по embeddings
    StorageContext,  # Связывает индекс с хранилищем
    get_response_synthesizer,  # Создает компонент генерации ответа
    load_index_from_storage,  # Загружает сохраненный индекс
)
from llama_index.core.ingestion import IngestionPipeline  # Обрабатывает документы
from llama_index.core.node_parser import SentenceSplitter  # Разбивает текст на nodes
from llama_index.core.prompts import PromptTemplate  # Шаблон промпта для LLM
from llama_index.core.query_engine import RetrieverQueryEngine  # Поиск и генерация ответа
from llama_index.core.vector_stores import (
    MetadataFilter,  # Одно условие фильтрации
    MetadataFilters,  # Набор условий для поиска
)
from llama_index.embeddings.huggingface import HuggingFaceEmbedding  # Локальные embeddings
from llama_index.llms.ollama import Ollama  # Подключение LLM через Ollama

# Работает при запуске из корня репозитория или из notebooks/.
ROOT = Path.cwd() if (Path.cwd() / "data" / "knowledge_base").is_dir() else Path.cwd().parent
DATA_DIR = ROOT / "data" / "knowledge_base"
if not DATA_DIR.is_dir():
    raise FileNotFoundError("Запустите Jupyter из корня репозитория или из notebooks/.")

FILES = [
    DATA_DIR / "HR_policy_2024.md",
    DATA_DIR / "HR_policy_2026.md",
    DATA_DIR / "HR_Remote_Work_2024_Archive.md",
    DATA_DIR / "HR_Remote_Work_2026_Advanced.md",
]
assert all(p.is_file() for p in FILES), "Не найдены документы для демо"
MODEL = "hf.co/Qwen/Qwen3-4B-GGUF:Q4_K_M"
EMBED_MODEL = "BAAI/bge-m3"
print("Документы:", DATA_DIR)

Документы: /Users/newuser/Documents/repo/rag-engineering-workshop/data/knowledge_base


## 1. Загрузка: файлы → Documents

`SimpleDirectoryReader` читает файлы и возвращает `Document`. Метаданные задаем по именам четырех файлов. В реальной системе год и статус лучше брать из надежного источника.

In [2]:
def file_metadata(file_path: str) -> dict:
    path = Path(file_path)
    year = 2024 if "2024" in path.name else 2026
    return {"filename": path.name, "year": year, "category": "HR"}

reader = SimpleDirectoryReader(
    input_files=[str(path) for path in FILES],
    file_metadata=file_metadata,
    filename_as_id=True,
)
documents = reader.load_data()
display(pd.DataFrame([
    {"id": doc.id_, **doc.metadata, "символов": len(doc.text)}
    for doc in documents
]))
print("Пример Document:\n", documents[0].text[:500])

,id,filename,year,category,символов
0,/Users/newuser/Documents/repo/rag-engineering-...,HR_policy_2024.md,2024,HR,2211
1,/Users/newuser/Documents/repo/rag-engineering-...,HR_policy_2026.md,2026,HR,2372
2,/Users/newuser/Documents/repo/rag-engineering-...,HR_Remote_Work_2024_Archive.md,2024,HR,605
3,/Users/newuser/Documents/repo/rag-engineering-...,HR_Remote_Work_2026_Advanced.md,2026,HR,774


Пример Document:
  # Политика Удаленной Работы (Архив 2024)

**ВНИМАНИЕ:** Этот документ относится к Архиву 2024 и предназначен для ссылки на историческую информацию о политике удаленной работы в корпорации РобоТех.

## I. Введение

В соответствии с измененными требованиями к работе и жизни в обществе, в нашей корпорации РобоТех был разработан и принят документ о политике удаленной работы. Целью данного документа является обеспечение соблюдения правил, регламентирующих полную удаленность для сотрудников на протяж


## 2. Ingestion: Documents → Nodes

`SentenceSplitter` разбивает текст по предложениям; размер и overlap задаются в токенах. Добавим его в `IngestionPipeline`. Метаданные документа переходят в каждый node.

In [3]:
splitter = SentenceSplitter(chunk_size=256, chunk_overlap=40)
pipeline = IngestionPipeline(transformations=[splitter])
nodes = pipeline.run(documents=documents)
print(f"Документов: {len(documents)}; nodes: {len(nodes)}")
node = nodes[0]
print("Тип:", type(node).__name__)
print("Node ID:", node.node_id)
print("Metadata:", node.metadata)
print("Связи:", list(node.relationships))
print("Текст:\n", node.text)

Документов: 4; nodes: 13
Тип: TextNode
Node ID: 9de2a351-6e1e-4a61-939e-2d1c391c8da1
Metadata: {'filename': 'HR_policy_2024.md', 'year': 2024, 'category': 'HR'}
Связи: [<NodeRelationship.SOURCE: '1'>, <NodeRelationship.NEXT: '3'>]
Текст:
 # Политика Удаленной Работы (Архив 2024)

**ВНИМАНИЕ:** Этот документ относится к Архиву 2024 и предназначен для ссылки на историческую информацию о политике удаленной работы в корпорации РобоТех.

## I. Введение

В соответствии с измененными требованиями к работе и жизни в обществе, в нашей корпорации РобоТех был разработан и принят документ о политике удаленной работы. Целью данного документа является обеспечение соблюдения правил, регламентирующих полную удаленность для сотрудников на протяжении 5 дней в неделю.

## II. Главные факты

1.


In [4]:
# Посмотрим, как один документ разделился на фрагменты.
example_name = "HR_policy_2026.md"
for i, node in enumerate((node for node in nodes if node.metadata["filename"] == example_name), 1):
    print(f"\n{'=' * 60}\nЧанк {i} | node_id: {node.node_id} | год: {node.metadata['year']}\n{'=' * 60}")
    print(node.text)


Чанк 1 | node_id: bbf0edd5-b4cd-497e-8239-2dc53a4ae5f9 | год: 2026
# Политика гибридной работы (Действует с 2026)

## Вступление

В соответствии с нашими корпоративными целями по улучшению эффективности и увеличению комфорта сотрудников, мы внедряем новую политику гибридной работы, которая регулирует обязанности и права сотрудников по работе как из офиса, так и удаленно. Данная политика действует с 2026 года.

## Главный текст

**Статус**: Активен

Наши сотрудники могут выбирать между гибридной работой или полностью удаленной работой в зависимости от их предпочтений и требований их должностей.

Чанк 2 | node_id: 37bd3fdb-5c37-4124-a4e5-8a47a7a9b0d9 | год: 2026
Однако, следует учесть, что сотрудники обязаны быть в офисе во Вторник, Среду, Четверг (из 5 рабочих дней недели). Дни Пн и Пт доступны для удаленной работы.

**Правила гибридной работы:**

- **Вход в офис**: Все сотрудники должны присутствовать во Вторник, Среду, Четверг с 9:00 до 18:00 по местному времени. При необходимости, м

## 3. Embeddings и индекс

`VectorStoreIndex` создает embeddings для nodes и записывает их во встроенный `SimpleVectorStore`. Индекс управляет доступом к данным, а хранилище сохраняет векторы и ищет по ним.

Используем ту же embedding-модель, что и в исходном ноутбуке. В разделе 8 подключим Qdrant и повторим поиск.

In [5]:
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
import torch

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print("Устройство:", device)
embed_model = HuggingFaceEmbedding(model_name=EMBED_MODEL, device=device, normalize=True)
llm = Ollama(model=MODEL, base_url="http://localhost:11434", temperature=0.1, request_timeout=600.0)
Settings.embed_model = embed_model
Settings.llm = llm

vector = embed_model.get_text_embedding("Удаленная работа")
print("Размер embedding:", len(vector))
print("Первые 5 значений:", vector[:5])

Устройство: mps
Размер embedding: 1024
Первые 5 значений: [0.015618442557752132, -0.0011712436098605394, -0.028278274461627007, 0.007271420676261187, -0.01450214721262455]


In [6]:
# Используем уже подготовленные nodes, чтобы не скрывать этап разбиения.
index = VectorStoreIndex(nodes, embed_model=embed_model, show_progress=True)
print("Тип vector store:", type(index.vector_store).__name__)
print("Индекс построен для", len(nodes), "nodes")

Generating embeddings:   0%|          | 0/13 [00:00<?, ?it/s]

Тип vector store: SimpleVectorStore
Индекс построен для 13 nodes


## 4. Retriever: что нашли до вызова LLM

`similarity_top_k` задает число возвращаемых nodes. `score` показывает близость, а не вероятность правильного ответа. Архив может попасть в выдачу; его позиция зависит от модели и запроса.

In [7]:
def show_nodes(results):
    if not results:
        print("Ничего не найдено.")
        return
    for i, item in enumerate(results, 1):
        node = item.node
        score = f"{item.score:.4f}" if item.score is not None else "нет"
        print(f"\n{'=' * 60}\nЧанк {i} | score: {score} | node_id: {node.node_id}")
        print(f"Файл: {node.metadata.get('filename')} | год: {node.metadata.get('year')}")
        print(f"Метаданные: {node.metadata}\n{'=' * 60}")
        print(node.text)

question = "Сколько дней в неделю можно работать из дома и в какие дни?"
retriever = index.as_retriever(similarity_top_k=4)
results = retriever.retrieve(question)
show_nodes(results)


Чанк 1 | score: 0.6195 | node_id: 37bd3fdb-5c37-4124-a4e5-8a47a7a9b0d9
Файл: HR_policy_2026.md | год: 2026
Метаданные: {'filename': 'HR_policy_2026.md', 'year': 2026, 'category': 'HR'}
Однако, следует учесть, что сотрудники обязаны быть в офисе во Вторник, Среду, Четверг (из 5 рабочих дней недели). Дни Пн и Пт доступны для удаленной работы.

**Правила гибридной работы:**

- **Вход в офис**: Все сотрудники должны присутствовать во Вторник, Среду, Четверг с 9:00 до 18:00 по местному времени. При необходимости, менеджеры могут изменить время работы для своей команды после согласования с HR-департаментом.
- **Удаленная работа**: Дни Пн и Пт предназначены для удаленной работы.

Чанк 2 | score: 0.5974 | node_id: cdb7e3ab-3b0e-49ae-865b-cc35d87bd20e
Файл: HR_Remote_Work_2026_Advanced.md | год: 2026
Метаданные: {'filename': 'HR_Remote_Work_2026_Advanced.md', 'year': 2026, 'category': 'HR'}
---
year: 2026
category: HR
department: PeopleOps
doc_type: policy
access_group: employees
graph_ready: 

## 5. Фильтр по метаданным

Ограничим поиск документами с `year=2026`. Фильтр работает, если метаданные заполнены верно, но сам по себе не проверяет актуальность документа.

In [8]:
filters = MetadataFilters(filters=[MetadataFilter(key="year", value=2026)])
filtered_retriever = index.as_retriever(similarity_top_k=4, filters=filters)
filtered_results = filtered_retriever.retrieve(question)
show_nodes(filtered_results)
assert all(item.node.metadata["year"] == 2026 for item in filtered_results)


Чанк 1 | score: 0.6195 | node_id: 37bd3fdb-5c37-4124-a4e5-8a47a7a9b0d9
Файл: HR_policy_2026.md | год: 2026
Метаданные: {'filename': 'HR_policy_2026.md', 'year': 2026, 'category': 'HR'}
Однако, следует учесть, что сотрудники обязаны быть в офисе во Вторник, Среду, Четверг (из 5 рабочих дней недели). Дни Пн и Пт доступны для удаленной работы.

**Правила гибридной работы:**

- **Вход в офис**: Все сотрудники должны присутствовать во Вторник, Среду, Четверг с 9:00 до 18:00 по местному времени. При необходимости, менеджеры могут изменить время работы для своей команды после согласования с HR-департаментом.
- **Удаленная работа**: Дни Пн и Пт предназначены для удаленной работы.

Чанк 2 | score: 0.5974 | node_id: cdb7e3ab-3b0e-49ae-865b-cc35d87bd20e
Файл: HR_Remote_Work_2026_Advanced.md | год: 2026
Метаданные: {'filename': 'HR_Remote_Work_2026_Advanced.md', 'year': 2026, 'category': 'HR'}
---
year: 2026
category: HR
department: PeopleOps
doc_type: policy
access_group: employees
graph_ready: 

## 6. Query Engine: найденные nodes → ответ

Query engine соединяет retriever и response synthesizer. В промпт передаем контекст и вопрос через `context_str` и `query_str`. Проверяйте ответ по источникам: модель может ошибаться.

In [9]:
qa_prompt = PromptTemplate("""Ты — ассистент компании РобоТех. Ответь на вопрос только по контексту.
Если в контексте нет ответа, скажи: «В документах нет информации об этом».
Не придумывай факты.

Контекст:
{context_str}

Вопрос: {query_str}
Ответ:""")

synthesizer = get_response_synthesizer(
    llm=llm, text_qa_template=qa_prompt, response_mode="compact"
)
query_engine = RetrieverQueryEngine(
    retriever=filtered_retriever, response_synthesizer=synthesizer
)
response = query_engine.query(question)
display(Markdown(str(response)))

Сотрудники могут работать из дома два дня в неделю: в понедельник и пятницу.

In [10]:
# Это nodes, переданные в синтез ответа, а не все документы индекса.
show_nodes(response.source_nodes)


Чанк 1 | score: 0.6195 | node_id: 37bd3fdb-5c37-4124-a4e5-8a47a7a9b0d9
Файл: HR_policy_2026.md | год: 2026
Метаданные: {'filename': 'HR_policy_2026.md', 'year': 2026, 'category': 'HR'}
Однако, следует учесть, что сотрудники обязаны быть в офисе во Вторник, Среду, Четверг (из 5 рабочих дней недели). Дни Пн и Пт доступны для удаленной работы.

**Правила гибридной работы:**

- **Вход в офис**: Все сотрудники должны присутствовать во Вторник, Среду, Четверг с 9:00 до 18:00 по местному времени. При необходимости, менеджеры могут изменить время работы для своей команды после согласования с HR-департаментом.
- **Удаленная работа**: Дни Пн и Пт предназначены для удаленной работы.

Чанк 2 | score: 0.5974 | node_id: cdb7e3ab-3b0e-49ae-865b-cc35d87bd20e
Файл: HR_Remote_Work_2026_Advanced.md | год: 2026
Метаданные: {'filename': 'HR_Remote_Work_2026_Advanced.md', 'year': 2026, 'category': 'HR'}
---
year: 2026
category: HR
department: PeopleOps
doc_type: policy
access_group: employees
graph_ready: 

### Проверка вопросом без ответа

В корпусе нет сведений о зарплате. Ожидаем отказ, но промпт его не гарантирует: сравните ответ с найденными фрагментами.

In [11]:
negative = query_engine.query("Какая зарплата у Senior Python Developer?")
display(Markdown(str(negative)))
show_nodes(negative.source_nodes)

В документах нет информации об этом


Чанк 1 | score: 0.3925 | node_id: e66e4784-5cfc-4995-907a-1f06e01c3a2e
Файл: HR_Remote_Work_2026_Advanced.md | год: 2026
Метаданные: {'filename': 'HR_Remote_Work_2026_Advanced.md', 'year': 2026, 'category': 'HR'}
Исключения согласуются через HR Business Partner и руководителя подразделения. Постоянная удаленная работа допускается только для сотрудников с письменным статусом Remote First.

Код политики для точного поиска: HR-REMOTE-2026. Внутренний короткий код: WFH-MON-FRI.

Владелец документа: департамент PeopleOps. Ответственная за политику: Мария Соколова.

Чанк 2 | score: 0.3643 | node_id: 7c7040b0-d405-460f-8a35-810f3fcfa794
Файл: HR_policy_2026.md | год: 2026
Метаданные: {'filename': 'HR_policy_2026.md', 'year': 2026, 'category': 'HR'}
- **Удаленная работа**: Дни Пн и Пт предназначены для удаленной работы. Сотрудники должны направить запрос о удаленном дне в HR-департамент не позднее чем за 2 дня до начала указанного периода.
- **Обязанности сотрудников**: Независимо от места ра

## 7. Сохранение и повторная загрузка индекса

Сохраним индекс на диск и загрузим его без повторной индексации. Временная папка будет удалена после выполнения ячейки. Для постоянного хранения укажите другую папку.

In [12]:
from tempfile import TemporaryDirectory

with TemporaryDirectory(prefix="llamaindex_demo_") as directory:
    index.storage_context.persist(persist_dir=directory)
    storage = StorageContext.from_defaults(persist_dir=directory)
    restored_index = load_index_from_storage(storage, embed_model=embed_model)
    restored_results = restored_index.as_retriever(similarity_top_k=2, filters=filters).retrieve(question)
    show_nodes(restored_results)


Чанк 1 | score: 0.6195 | node_id: 37bd3fdb-5c37-4124-a4e5-8a47a7a9b0d9
Файл: HR_policy_2026.md | год: 2026
Метаданные: {'filename': 'HR_policy_2026.md', 'year': 2026, 'category': 'HR'}
Однако, следует учесть, что сотрудники обязаны быть в офисе во Вторник, Среду, Четверг (из 5 рабочих дней недели). Дни Пн и Пт доступны для удаленной работы.

**Правила гибридной работы:**

- **Вход в офис**: Все сотрудники должны присутствовать во Вторник, Среду, Четверг с 9:00 до 18:00 по местному времени. При необходимости, менеджеры могут изменить время работы для своей команды после согласования с HR-департаментом.
- **Удаленная работа**: Дни Пн и Пт предназначены для удаленной работы.

Чанк 2 | score: 0.5974 | node_id: cdb7e3ab-3b0e-49ae-865b-cc35d87bd20e
Файл: HR_Remote_Work_2026_Advanced.md | год: 2026
Метаданные: {'filename': 'HR_Remote_Work_2026_Advanced.md', 'year': 2026, 'category': 'HR'}
---
year: 2026
category: HR
department: PeopleOps
doc_type: policy
access_group: employees
graph_ready: 

## 8. Вариант с Qdrant

Повторим индексацию и поиск с теми же nodes и embedding-моделью. Qdrant хранит векторы и метаданные в коллекции; LlamaIndex обращается к ней через `QdrantVectorStore`.

Запустите Docker, затем в терминале из корня репозитория:

```bash
docker compose up -d qdrant
```

Qdrant доступен на `http://localhost:6333`, а данные сохраняются в Docker volume. Для ноутбука используем отдельную коллекцию `robotex_notebook_demo`.

In [13]:
from qdrant_client import QdrantClient
from llama_index.vector_stores.qdrant import QdrantVectorStore  # Адаптер LlamaIndex к Qdrant

QDRANT_URL = "http://localhost:6333"
COLLECTION = "robotex_notebook_demo"
client = QdrantClient(url=QDRANT_URL)

# Если коллекция уже есть, подключаемся к ней без повторной индексации.
# Для новой версии документов или модели создайте новую коллекцию.
vector_store = QdrantVectorStore(client=client, collection_name=COLLECTION)
if client.collection_exists(COLLECTION):
    qdrant_index = VectorStoreIndex.from_vector_store(vector_store, embed_model=embed_model)
    print(f"Подключена существующая коллекция: {COLLECTION}")
    print("Она может содержать данные предыдущего запуска.")
else:
    qdrant_storage = StorageContext.from_defaults(vector_store=vector_store)
    qdrant_index = VectorStoreIndex(
        nodes, storage_context=qdrant_storage, embed_model=embed_model, show_progress=True
    )
    print(f"Создана коллекция: {COLLECTION}")

Подключена существующая коллекция: robotex_notebook_demo
Она может содержать данные предыдущего запуска.


### Поиск с фильтром в Qdrant

Используем тот же вопрос и фильтр `year=2026`. Результаты могут отличаться от поиска в памяти, особенно если коллекция осталась от предыдущего запуска.

In [14]:
qdrant_retriever = qdrant_index.as_retriever(similarity_top_k=4, filters=filters)
qdrant_results = qdrant_retriever.retrieve(question)
show_nodes(qdrant_results)


Чанк 1 | score: 0.6195 | node_id: bdcc669e-cd35-423d-b110-5d22e9d6d0ba
Файл: HR_policy_2026.md | год: 2026
Метаданные: {'filename': 'HR_policy_2026.md', 'year': 2026, 'category': 'HR'}
Однако, следует учесть, что сотрудники обязаны быть в офисе во Вторник, Среду, Четверг (из 5 рабочих дней недели). Дни Пн и Пт доступны для удаленной работы.

**Правила гибридной работы:**

- **Вход в офис**: Все сотрудники должны присутствовать во Вторник, Среду, Четверг с 9:00 до 18:00 по местному времени. При необходимости, менеджеры могут изменить время работы для своей команды после согласования с HR-департаментом.
- **Удаленная работа**: Дни Пн и Пт предназначены для удаленной работы.

Чанк 2 | score: 0.5974 | node_id: b327f6bf-8960-40de-8166-09c1d023fe38
Файл: HR_Remote_Work_2026_Advanced.md | год: 2026
Метаданные: {'filename': 'HR_Remote_Work_2026_Advanced.md', 'year': 2026, 'category': 'HR'}
---
year: 2026
category: HR
department: PeopleOps
doc_type: policy
access_group: employees
graph_ready: 

### Ответ по данным Qdrant

Меняется retriever, а synthesizer и промпт остаются прежними. Ячейка обращается к Ollama.

In [15]:
qdrant_query_engine = RetrieverQueryEngine(
    retriever=qdrant_retriever, response_synthesizer=synthesizer
)
qdrant_response = qdrant_query_engine.query(question)
display(Markdown(str(qdrant_response)))
show_nodes(qdrant_response.source_nodes)

Сотрудники могут работать из дома два дня в неделю: в понедельник и пятницу.


Чанк 1 | score: 0.6195 | node_id: bdcc669e-cd35-423d-b110-5d22e9d6d0ba
Файл: HR_policy_2026.md | год: 2026
Метаданные: {'filename': 'HR_policy_2026.md', 'year': 2026, 'category': 'HR'}
Однако, следует учесть, что сотрудники обязаны быть в офисе во Вторник, Среду, Четверг (из 5 рабочих дней недели). Дни Пн и Пт доступны для удаленной работы.

**Правила гибридной работы:**

- **Вход в офис**: Все сотрудники должны присутствовать во Вторник, Среду, Четверг с 9:00 до 18:00 по местному времени. При необходимости, менеджеры могут изменить время работы для своей команды после согласования с HR-департаментом.
- **Удаленная работа**: Дни Пн и Пт предназначены для удаленной работы.

Чанк 2 | score: 0.5974 | node_id: b327f6bf-8960-40de-8166-09c1d023fe38
Файл: HR_Remote_Work_2026_Advanced.md | год: 2026
Метаданные: {'filename': 'HR_Remote_Work_2026_Advanced.md', 'year': 2026, 'category': 'HR'}
---
year: 2026
category: HR
department: PeopleOps
doc_type: policy
access_group: employees
graph_ready: 

## Что попробовать

- Изменить `chunk_size` и `chunk_overlap`, повторить ячейки начиная с ingestion и сравнить фрагменты.
- Поменять `similarity_top_k` и посмотреть на источники ответа.
- Убрать фильтр или задать `year=2024` и сравнить выдачу.
- Подключить весь каталог, предварительно определив метаданные для остальных документов.

Hybrid retrieval, reranking и router в базовый пример не входят.